# Part 2 · Normalization

**Objective.** Analyze the dependencies in the source tables, decide how far to normalize, and build and load a trusted `core` schema with dbt.

**Before you start.** Part 1 is reviewed and approved.

**Tickets in this part**

| Ticket | Title | Type | Estimate |
|---|---|---|---|
| DM-201 | Keys and dependencies | Analysis | 1 day |
| DM-202 | Assess 1NF to BCNF | Analysis | 1 day |
| DM-203 | Assess 4NF and 5NF | Analysis | 1 day |
| DM-204 | Design the `core` schema | Design | 1 day |
| DM-205 | Implement `core` as dbt models | Engineering | 1 day |
| DM-206 | Transform `raw` to `core` in dbt | Engineering | 2 days |
| DM-207 | Data quality tests in dbt | Testing | 1 day |

**Ground rules**
- Work on one ticket at a time, in order. Stop at every checkpoint.
- Back every claim about the data with a query.
- Structure changes go in numbered migrations (raw, audit) or in dbt models (core, dwh) only.
- Update your findings log and decisions log as you go.
- Project brief: the first cells of notebook `00_setup_and_raw_load`.

## DM-201 · Keys and dependencies
**Type:** Analysis · **Priority:** High · **Estimate:** 1 day

**Context**  
Normalization is decided by the dependencies that really hold in the data and in the business, not by how the sheets look.

**What to do**
- For every raw table, identify the candidate keys and list the functional dependencies you believe hold.
- Test each dependency with a SQL query that looks for counter-examples.

**Deliverables**
- `docs/dependencies.md`
- Test queries in `sql/checks/`

**Acceptance criteria**
- [ ] Each dependency has a counter-example query and its result
- [ ] Dependencies that may hold only by coincidence of this sample are marked and questioned
- [ ] Composite keys are examined for partial dependencies, and non-key columns are examined for dependencies on other non-key columns

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-201.** Share your work for review before you start the next ticket.

## DM-202 · Assess 1NF to BCNF
**Type:** Analysis · **Priority:** High · **Estimate:** 1 day

**Context**  
You need a clear record of where every table stands before you change anything.

**What to do**
- For each raw table, state the highest normal form it satisfies, the next rule it fails, and the evidence.
- Propose a decomposition wherever a rule fails.

**Deliverables**
- `docs/normalization_log.md` with columns: table, current normal form, violated rule, evidence, proposed decomposition

**Acceptance criteria**
- [ ] All raw tables are assessed
- [ ] 1NF is examined at column level (atomic values and repeating groups)
- [ ] Each proposed decomposition lists the resulting tables and their keys

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-202.** Share your work for review before you start the next ticket.

## DM-203 · Assess 4NF and 5NF
**Type:** Analysis · **Priority:** High · **Estimate:** 1 day

**Context**  
Higher normal forms matter only when the data and the business rules require them. You must be able to show which case you are in.

**What to do**
- Determine whether any table has multivalued dependencies or join dependencies.
- Test them with projection-and-rejoin queries.
- Decide whether to decompose, and state the business rules each decision depends on.

**Deliverables**
- Section in `docs/normalization_log.md`
- Open questions for any decision that depends on a business rule

**Acceptance criteria**
- [ ] Every table has a recorded conclusion for 4NF and for 5NF, not just "not applicable"
- [ ] Every decomposition is proven lossless by a query (the rejoined result equals the original)
- [ ] Each decision that depends on a business rule is logged with the rule stated

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-203.** Share your work for review before you start the next ticket.

## DM-204 · Design the `core` schema
**Type:** Design · **Priority:** High · **Estimate:** 1 day

**Context**  
`core` is the trusted, normalized layer. Everything downstream is built from it.

**What to do**
- Design the `core` schema to at least 3NF, and further where justified.
- Define each table with its keys, foreign keys, and constraints. For each dbt model, decide its materialization (view, table, or incremental) and justify.
- Decide which rules dbt enforces as constraints (model contracts) and which you check with tests.
- For each repeated value, decide whether it is true redundancy or a value that must be kept as it is, and justify.
- Update the logical ERD.

**Deliverables**
- `docs/erd/core_schema.*`
- Updated `docs/erd/logical.*`
- Decisions in `docs/decisions.md`

**Acceptance criteria**
- [ ] Every constraint is justified by a rule you found earlier
- [ ] Reference (lookup) data is handled consistently
- [ ] Every decision to keep or remove a repeated value has a written reason
- [ ] Every model has a documented materialization with a reason

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-204.** Share your work for review before you start the next ticket.

## DM-205 · Implement `core` as dbt models
**Type:** Engineering · **Priority:** High · **Estimate:** 1 day

**Context**  
From here on, the structure and the data of `core` are built by dbt, so the same code can rebuild it anywhere.

**What to do**
- Create the `core` models in `dbt/models/core/`, with YAML files that describe every model and column.
- Add the keys and constraints you designed.
- Run `dbt build --select core` and `dbt docs generate`.

**Deliverables**
- `dbt/models/core/*.sql` and `*.yml`

**Acceptance criteria**
- [ ] A development database rebuilt from scratch (raw load plus dbt) produces the same `core` structure
- [ ] At least 5 negative tests prove that the rules reject bad data
- [ ] Every model and column has a description
- [ ] The lineage graph in the dbt docs shows the raw sources flowing into `core`
- [ ] There is no manual DDL

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-205.** Share your work for review before you start the next ticket.

## DM-206 · Transform `raw` to `core` in dbt
**Type:** Engineering · **Priority:** High · **Estimate:** 2 days

**Context**  
`raw` holds what the source systems delivered, including anything wrong in it. `core` must contain only data you can trust, and every change must be traceable.

**What to do**
- Write the dbt models that fill `core` from `raw` (cleaning, type conversion, de-duplication, key mapping, and so on).
- Anything excluded or changed must be traceable.

**Deliverables**
- dbt models in `dbt/models/core/`
- A quarantine (rejected records) model
- Cleansing rules in `docs/decisions.md`

**Acceptance criteria**
- [ ] The quarantine model stores each excluded record with the rule and the reason
- [ ] Per table, raw rows = core rows + rejected + merged, with no unexplained difference
- [ ] Each cleansing rule is documented with its business justification, or logged as an open question
- [ ] Running `dbt build` twice gives the same result

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-206.** Share your work for review before you start the next ticket.

## DM-207 · Data quality tests in dbt
**Type:** Testing · **Priority:** High · **Estimate:** 1 day

**Context**  
A pipeline without tests is a pipeline you cannot trust. These tests will also run again in production.

**What to do**
- Write dbt tests: generic tests (for example unique, not null, relationships, accepted values) and singular tests (custom SQL in `dbt/tests/`).
- Cover completeness, uniqueness, referential integrity, valid values, cross-table reconciliation, and business rules.
- Set the severity (warn or error) of each test and explain your choice.

**Deliverables**
- Tests in `dbt/models/**/*.yml` and `dbt/tests/`

**Acceptance criteria**
- [ ] At least 25 tests, including at least 8 singular tests
- [ ] Every test has a clear name or description and a severity
- [ ] `dbt test` finishes in under a minute and can be re-run
- [ ] Every failure is either fixed or logged as a known issue with an owner

In [ ]:
# Your work here

**Notes and decisions**

*(write here)*

> 🛑 **Checkpoint DM-207.** Share your work for review before you start the next ticket.